In [25]:
import pandas as pd

diplome = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\diplome_region.csv")
chomage = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\chomage_format_long.csv")
creation_per_1000 = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\creation_per_1000.csv")
salaires = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\salaires.csv", sep=';')
population = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\population_par_region_annee.csv")

chomage=chomage.rename(columns={'Region':'region_nom', 'TIME_VALUE':'TIME_PERIOD'})

salaires=salaires.rename(columns={'REGION_NOM':'region_nom', 'ANNEE1':'TIME_PERIOD'})


1. L'Hypothèse de Stabilité (La méthode la plus courante)
Le niveau d'éducation d'une région change très lentement. On peut raisonnablement supposer que si une région est très diplômée en 2022, elle l'était aussi (relativement aux autres) en 2015.

La méthode : Tu dupliques la valeur de 2022 sur toutes les années pour chaque région. Techniquement, tu fais un merge sur la colonne region_nom. Pandas va automatiquement répéter la valeur unique du diplôme 2022 sur les 10 lignes (années) de chaque région.

Conséquence statistique : Cette variable expliquera les différences entre les régions (pourquoi l'Île-de-France crée plus d'entreprises que la Creuse), mais elle n'expliquera pas les variations annuelles (pourquoi 2018 est meilleur que 2017).

In [35]:

regression = pd.merge(creation_per_1000, diplome[['Pourcentage_diplomes_superieur', 'Pourcentage_aucun_diplome_ou_certificat_d_etudes_primaires','Pourcentage_Baccalauréat / Brevet professionnel ou équivalent', 'region_nom']], on='region_nom', how='left')

regression = pd.merge(regression, chomage[['Taux de chômage par région', 'region_nom', 'TIME_PERIOD']], on=['region_nom', 'TIME_PERIOD'], how='left') 

regression=pd.merge(regression, population[['variation_population_pourcentage', 'densité de population', 'region_nom', 'TIME_PERIOD']], on=['region_nom', 'TIME_PERIOD'], how='left')


#on va garder uniquement les colonnes utiles pour la régression
#fait le choix de garder 'Pourcentage_diplomes_superieur' pour la variable sur les diplômes
regression_finale=regression[['creations_per_1000', 'Pourcentage_diplomes_superieur', 'Taux de chômage par région', 'variation_population_pourcentage', 'densité de population']]

#envoyer regression_finale en csv
regression_finale.to_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\regression1.csv", index=False)


Afin d'identifier les facteurs influençant le dynamisme entrepreneurial des régions, nous procédons à une régression linéaire multiple (Méthode des Moindres Carrés Ordinaires - OLS).Le modèle économétrique testé est le suivant :$$TauxCréation_i = \beta_0 + \beta_1 \cdot Diplôme + \beta_2 \cdot Chômage + \beta_3 \cdot VariationPop + \beta_4 \cdot Densité + \epsilon_i$$

Choix des variables :Variable expliquée ($Y$) : Le taux de création d'entreprises (pour 1000 habitants), indicateur direct du dynamisme entrepreneurial.

Variables explicatives ($X$) :
- Capital Humain : Le pourcentage de diplômés du supérieur (proxy du niveau de qualification de la région).
- Conjoncture Économique : Le taux de chômage (pour tester l'effet "push" vs "pull").
- Dynamisme Démographique : La variation de la population (attractivité du territoire)
- Agglomération : La densité de population (effets de métropolisation).

Hypothèse de modélisation : Nous utilisons une approche "Pooled OLS" sur l'ensemble de notre panel (130 observations : 13 régions $\times$ 10 années). Cette méthode permet d'inclure des variables structurelles invariantes sur la période étudiée (comme le niveau de diplôme, fixé ici à son niveau de 2022) et de mesurer leur impact global sur les disparités inter-régionales.

In [33]:
import statsmodels.api as sm
import pandas as pd

# 1. Définition des variables
# Y = La variable cible (à expliquer)
y = regression_finale['creations_per_1000']

# X = Les variables explicatives
features = [
    'Taux de chômage par région', 
    'Pourcentage_diplomes_superieur',     
    'variation_population_pourcentage', 
    'densité de population'
]
X = regression_finale[features]

# 2. Ajout de la constante (Intercept)
# C'est crucial : sans ça, la droite de régression est forcée de passer par 0
X = sm.add_constant(X)

# 3. Création et ajustement du modèle (MCO = Moindres Carrés Ordinaires)
model = sm.OLS(y, X)
results = model.fit()

# 4. Affichage du rapport complet
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:     creations_per_1000   R-squared:                       0.577
Model:                            OLS   Adj. R-squared:                  0.563
Method:                 Least Squares   F-statistic:                     42.58
Date:              lun., 15 déc. 2025   Prob (F-statistic):           1.70e-22
Time:                        21:57:23   Log-Likelihood:                -316.14
No. Observations:                 130   AIC:                             642.3
Df Residuals:                     125   BIC:                             656.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   

1. Performance globale du modèle. Qualité de l'ajustement ($R^2$) : Le $R^2$ est de 0.577. Cela signifie que notre modèle explique 57,7 % de la variance du taux de création d'entreprises. 
Pour des données socio-économiques, c'est un score élevé, indiquant que les variables choisies sont très pertinentes pour expliquer le phénomène.Significativité globale (F-statistic) : La probabilité associée (Prob F-statistic: $1.70e-22$) est quasi nulle, ce qui confirme que le modèle dans son ensemble est statistiquement valide.

2. Analyse des déterminants (Variables significatives). Trois variables ont un impact statistiquement significatif (P-value < 0.05) sur la création d'entreprise :
- Le Taux de Chômage (Coef: -0.82, P<0.001) :Impact : Il existe une relation négative forte. Toutes choses égales par ailleurs, une augmentation de 1 point du taux de chômage entraîne une baisse de 0.82 point du taux de création.Interprétation : Ce résultat valide l'effet "Pull" (opportunité). La création d'entreprise est favorisée par une bonne santé économique. L'hypothèse de l'entrepreneuriat de nécessité (effet "Push"), où le chômage pousserait à la création, est ici rejetée.
- Le Capital Humain / Diplômes (Coef: +0.53, P<0.001) :Impact : Relation positive. Une augmentation de 1 point du pourcentage de diplômés du supérieur augmente le taux de création de 0.53 point.Interprétation : Le niveau de qualification d'un territoire est un moteur essentiel de l'entrepreneuriat.
- Le Dynamisme Démographique (Coef: +2.72, P=0.003) :Impact : Relation positive très forte. Une région dont la population croît attire les créateurs. C'est un indicateur d'attractivité du territoire (demande locale croissante).

3. La variable non-significative. La Densité de population (P=0.540) :Avec une P-value de 0.54 (bien supérieure au seuil de 0.05), nous ne pouvons pas conclure que la densité de population a un impact direct sur la création d'entreprises, une fois que l'on contrôle déjà pour le niveau d'éducation et le chômage. L'effet "métropole" passe probablement déjà par la variable des diplômés.

Note Méthodologique (Limites)Le test de Durbin-Watson est de 0.410, ce qui est faible (loin de la valeur cible de 2). Cela indique une présence d'autocorrélation positive dans les résidus. C'est attendu dans ce type de modèle "Pooled OLS" où les mêmes régions sont suivies sur plusieurs années. Bien que cela n'invalide pas les coefficients (la tendance reste vraie), cela suggère que la précision des intervalles de confiance pourrait être surestimée.

Idée ça peut être intéressant de prendre en compte l'âge d'une région en créant une variable binaire par exemple.